In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import plot_model

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
# Initialize random seeds for reproducibility

dictionary = pd.read_csv('FINANCE384_stock_month_data_dictionary.csv')
display(dictionary)

/Users/thismac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,variable,description,role
0,date,End of month t (CRSP month-end trading date),identifier; not a predictor
1,permno,CRSP permanent security identifier,identifier; not a predictor
2,ticker,Exchange ticker at month t,identifier; not a predictor
3,comnam,Company name at month t,identifier; not a predictor
4,industry,Coarse SIC division label (Agriculture collaps...,"classification, known at t; permitted as a pre..."
5,ff49_code,Fama-French 49-industry code (Ships collapsed ...,"classification, known at t; permitted as a pre..."
6,ff49_name,Fama-French 49-industry name (Ships collapsed ...,"classification, known at t; permitted as a pre..."
7,gvkey,Compustat firm identifier (missing if no CCM l...,identifier; not a predictor
8,size,log market capitalization (USD) at end of mont...,"characteristic, contemporaneous with t"
9,bm,Book equity (6-month availability lag) / marke...,"characteristic, contemporaneous with t"


In [2]:
determined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, contemporaneous with t", "variable"].tolist()

predetermined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, predetermined at t-1", "variable"].tolist()

determined_classifications = dictionary.loc[
    dictionary["role"] == "classification, known at t; permitted as a predictor if encoded", "variable"].tolist()

print(f"Rows: {len(dictionary):,}")
print(f"Determined characteristics: {determined_characteristics}")
print(f"Predetermined characteristics: {predetermined_characteristics}")
print(f"Determined classifications: {determined_classifications}")

FEATURES = determined_characteristics + predetermined_characteristics
print("Number of features:", len(FEATURES))

Rows: 27
Determined characteristics: ['size', 'bm', 'vol12', 'beta60', 'ivol60', 'turnover', 'dollar_volume', 'amihud_illiq', 'divyield', 'mkt_12m', 'mkt_vol_12m', 'down_market']
Predetermined characteristics: ['mom12_2', 'gross_profit', 'roe', 'asset_growth', 'leverage', 'accruals']
Determined classifications: ['industry', 'ff49_code', 'ff49_name']
Number of features: 18


In [3]:
panel = pd.read_csv('FINANCE384_assignmentA_development_panel.csv', parse_dates=['date'])

print(f"Rows           : {len(panel):,}")
print(f"Distinct stocks: {panel['permno'].nunique():,}")
print(f"Months         : {panel['date'].nunique():,} "
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
panel.head()


print(f"Duplicate (permno, date) rows: {panel.duplicated(['permno', 'date']).sum()}")

# Check 2: how many stocks are in each month, and how many months does a stock survive?
stocks_per_month = panel.groupby("date")["permno"].nunique()
months_per_stock = panel.groupby("permno")["date"].nunique()
print(f"Stocks per month : {stocks_per_month.min()} to {stocks_per_month.max()} "
      f"(mean {stocks_per_month.mean():.0f})")
print(f"Months per stock : {months_per_stock.min()} to {months_per_stock.max()} "
      f"(median {months_per_stock.median():.0f})")


Rows           : 198,298
Distinct stocks: 1,252
Months         : 396 (1990-01 to 2022-12)
Duplicate (permno, date) rows: 0
Stocks per month : 497 to 505 (mean 501)
Months per stock : 1 to 396 (median 122)


In [4]:
TARGET = "target_ret_excess_tp1"

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[["permno", "month", "ret_excess_t"]]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(next_month_return, on=["permno", "month"], how="left", validate="one_to_one")

final_month = (
    panel.groupby("permno")["month"]
    .transform("max")
    .eq(panel["month"])
)

no_target = panel[TARGET].isna()
# Diagnostics
print(f"Rows: {len(panel):,}")
print(f"Usable targets: {panel[TARGET].notna().sum():,}")
print(f"No month t+1 in data: {no_target.sum():,}")
print(f"Stock's last month: {(no_target & final_month).sum():,}")
print(f"Gap in coverage: {(no_target & ~final_month).sum():,}")

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())

print("\nFeature summary:")
display(
    panel[FEATURES + [TARGET]]
    .describe()
    .loc[["mean", "std", "min", "max"]]
)

panel[["permno", "month", "ret_excess_t", TARGET]].head()

Rows: 198,298
Usable targets: 197,016
No month t+1 in data: 1,282
Stock's last month: 1,252
Gap in coverage: 30

Missing values:
size                         0
bm                        4242
vol12                     5268
beta60                   17211
ivol60                   17211
turnover                     0
dollar_volume                0
amihud_illiq                 0
divyield                  5927
mkt_12m                      0
mkt_vol_12m                  0
down_market                  0
mom12_2                   6023
gross_profit               515
roe                       4259
asset_growth               737
leverage                   467
accruals                  6615
target_ret_excess_tp1     1282
dtype: int64

Feature summary:


,size,bm,vol12,beta60,ivol60,turnover,dollar_volume,amihud_illiq,divyield,mkt_12m,mkt_vol_12m,down_market,mom12_2,gross_profit,roe,asset_growth,leverage,accruals,target_ret_excess_tp1
mean,23.0467,0.5098,0.0847,1.0445,0.0725,0.1647,3.3957e+09,0.0003,0.0207,0.0891,0.0404,0.2297,0.1158,0.2994,0.1733,0.1099,0.2625,-0.0482,0.0083
std,1.3306,0.4449,0.0452,0.5579,0.0304,0.1518,6.9844e+09,0.0018,0.0196,0.1661,0.0164,0.4207,0.3274,0.2254,0.2518,0.2472,0.1713,0.0555,0.0990
min,17.3590,0.0082,0.0193,-0.3211,0.0257,0.0104,9.6556e+05,0.0000,0.0000,-0.4330,0.0108,0.0000,-0.9108,-0.1985,-0.8534,-0.4729,0.0000,-0.6748,-0.8703
max,27.5092,11.2483,0.5192,4.0973,0.2498,2.1366,2.3617e+11,0.1307,0.5481,0.6466,0.0894,1.0000,4.2891,1.1964,4.5006,3.0477,1.0184,0.1612,2.6266


,permno,month,ret_excess_t,target_ret_excess_tp1
0,10057,1990-01,-0.0540,-0.0179
1,10057,1990-02,-0.0179,-0.0558
2,10057,1990-03,-0.0558,-0.0485
3,10057,1990-04,-0.0485,-0.0205
4,10057,1990-05,-0.0205,-0.1035


In [5]:
industry = 'ff49_name'

for feature in FEATURES:
    monthly_median = (
        panel.groupby('date')[feature]
        .transform('median'))
    
    panel[feature] = (
        panel[feature]
        .fillna(monthly_median))

panel = panel.loc[panel[TARGET].notna()].copy()

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())


Missing values:
size                     0
bm                       0
vol12                    0
beta60                   0
ivol60                   0
turnover                 0
dollar_volume            0
amihud_illiq             0
divyield                 0
mkt_12m                  0
mkt_vol_12m              0
down_market              0
mom12_2                  0
gross_profit             0
roe                      0
asset_growth             0
leverage                 0
accruals                 0
target_ret_excess_tp1    0
dtype: int64


In [6]:
train = panel.loc[panel['month'].between('1990-01', '2014-12')].copy()
valid = panel.loc[panel['month'].between('2015-01', '2018-12')].copy()
test = panel.loc[panel['month'].between('2019-01', '2022-11')].copy()
train_valid = pd.concat([train, valid]).copy()

split_summary = pd.DataFrame([
    {'sample': name, 'months': block['month'].nunique(), 'rows': len(block),
     'start': str(block['month'].min()), 'end': str(block['month'].max())}
    for name, block in [('Train', train), ('Validation', valid), ('Test', test)]
])
display(split_summary)


,sample,months,rows,start,end
0,Train,300,149334,1990-01,2014-12
1,Validation,48,24051,2015-01,2018-12
2,Test,47,23631,2019-01,2022-11


In [7]:
def standardize(block, fitted_on):
    return (
        block[FEATURES]
        - fitted_on[FEATURES].mean()
    ) / fitted_on[FEATURES].std(ddof=0)


X_train = standardize(train, fitted_on=train)
X_valid = standardize(valid, fitted_on=train)
X_test = standardize(test, fitted_on=train)
X_train_valid = standardize(train_valid, fitted_on=train)

y_train, y_valid, y_test = train[TARGET], valid[TARGET], test[TARGET]
y_train_valid = train_valid[TARGET]

ols_benchmark = LinearRegression().fit(
    X_train_valid,
    y_train_valid
)

print("Number of coefficients:", len(ols_benchmark.coef_))
print("Intercept:", ols_benchmark.intercept_)

Number of coefficients: 18
Intercept: 0.00888480079326484


In [8]:
def neural_network(input):
    model = Sequential()
    model.add(
        Dense(
            16,
            input_shape=(input,),
            activation="relu"))
    model.add(
        Dense(
            1,
            activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mean_squared_error")
    model.summary()
    return model


In [9]:

def run_network(batch_size, SEED, EPOCHS):

    # Ensure every experiment begins from the same random seed
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    # Create a fresh network
    model = neural_network(len(FEATURES))

    # Train the network
    history = model.fit(
        X_train.values,
        y_train.values,
        validation_data=(
            X_valid.values,
            y_valid.values
        ),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            )
        ],
        verbose=1,
        shuffle=False
    )

    # Create the error table
    error_table = pd.DataFrame({
        "Batch Size": batch_size,
        "Seed": SEED,
        "Epoch": range(
            1,
            len(history.history["loss"]) + 1
        ),
        "Training MSE": history.history["loss"],
        "Validation MSE": history.history["val_loss"]
    })
    error_table["Training RMSE"] = np.sqrt(
        error_table["Training MSE"]
    )

    error_table["Validation RMSE"] = np.sqrt(
        error_table["Validation MSE"]
    )
    print(f"Batch size: {batch_size}, "
        f"Seed: {SEED}, "
        f"Maximum epochs: {EPOCHS}\n")
    display(error_table.set_index("Epoch"))

    best_row = error_table.loc[
    error_table["Validation RMSE"].idxmin()]
    model_results.append(best_row)



In [10]:
# comparing batch sizes
model_results = []

run_network(32, 200, 100)   
run_network(16, 200, 100)
model_results = pd.DataFrame(model_results)
display(model_results.sort_values(by="Validation RMSE").reset_index(drop=True))

/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

...

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 252us/step - loss: 0.2391 - val_loss: 0.0076
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 248us/step - loss: 0.0103 - val_loss: 0.0066
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 261us/step - loss: 0.0094 - val_loss: 0.0065
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 252us/step - loss: 0.0092 - val_loss: 0.0064
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 247us/step - loss: 0.0091 - val_loss: 0.0065
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 248us/step - loss: 0.0090 - val_loss: 0.0065
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 249us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 263us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 247us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 249us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 11/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 274us/step - loss: 0.0089 - val_loss: 0.00

,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,
1,16,200,0.0678,0.0076,0.2605,0.0874
2,16,200,0.0111,0.0066,0.1055,0.0810
3,16,200,0.0105,0.0065,0.1023,0.0805
4,16,200,0.0103,0.0064,0.1013,0.0801
5,16,200,0.0101,0.0065,0.1007,0.0804
6,16,200,0.0101,0.0065,0.1004,0.0803
7,16,200,0.0101,0.0064,0.1003,0.0801
8,16,200,0.0100,0.0064,0.1002,0.0798
9,16,200,0.0100,0.0064,0.1002,0.0799


,Batch Size,Seed,Epoch,Training MSE,Validation MSE,Training RMSE,Validation RMSE
0,16.0,200.0,29.0,0.0099,0.0062,0.0994,0.0790
1,32.0,200.0,15.0,0.0099,0.0063,0.0996,0.0793


In [11]:
EPOCHS = 100
BATCH_SIZE = 16
SEED = 200

def neural_network(input, hidden_layers, learning_rate):
    model = Sequential()

    # First hidden layer
    model.add(
        Dense(
            hidden_layers[0],
            input_shape=(input,),
            activation="relu"))

    # Any additional hidden layers
    for neurons in hidden_layers[1:]:
        model.add(
            Dense(
                neurons,
                activation="relu"))

    # One predicted return
    model.add(
        Dense(
            1,
            activation="linear"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mean_squared_error")
    return model


def run_network(hidden_layers, learning_rate):

    # Give every candidate the same starting seed
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    # Create a fresh network
    model = neural_network(
        len(FEATURES),
        hidden_layers,
        learning_rate)

    # Train the network
    history = model.fit(
        X_train.values,
        y_train.values,
        validation_data=(
            X_valid.values,
            y_valid.values
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True)
        ],
        verbose=1,
        shuffle=False)

    # Store the error from every completed epoch
    error_table = pd.DataFrame({
        "Hidden Layers": str(hidden_layers),
        "Learning Rate": learning_rate,
        "Batch Size": BATCH_SIZE,
        "Seed": SEED,
        "Epoch": range(
            1,
            len(history.history["loss"]) + 1
        ),
        "Training MSE": history.history["loss"],
        "Validation MSE": history.history["val_loss"]
    })
    error_table["Training RMSE"] = np.sqrt(
        error_table["Training MSE"])
    error_table["Validation RMSE"] = np.sqrt(
        error_table["Validation MSE"])
            
    # Display the complete table
    display(
        error_table
        .set_index("Epoch")
        .round(6))


    best_row = error_table.loc[
    error_table["Validation RMSE"].idxmin()]
    model_results.append(best_row)

    return model, history, error_table


In [12]:
model_results = []
run_network([32, 16, 8], 0.0015)
run_network([32, 16, 8], 0.001)
run_network([32, 16, 8], 0.0005)

run_network([16, 8], 0.0015)
run_network([16, 8], 0.001)
run_network([16, 8], 0.0005)

run_network([32], 0.001)
run_network([16], 0.001)


model_results = pd.DataFrame(model_results)
display(
    model_results
    .sort_values("Validation RMSE")
    .reset_index(drop=True)
)


Epoch 1/100


/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 305us/step - loss: 0.0098 - val_loss: 0.0066
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 281us/step - loss: 0.0090 - val_loss: 0.0063
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 278us/step - loss: 0.0089 - val_loss: 0.0062
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 284us/step - loss: 0.0089 - val_loss: 0.0062
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 279us/step - loss: 0.0089 - val_loss: 0.0063
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 293us/step - loss: 0.0088 - val_loss: 0.0062
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 280us/step - loss: 0.0088 - val_loss: 0.0064
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 279us/step - loss: 0.0087 - val_loss: 0.0064
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 289us/step - loss: 0.0087 - val_loss: 0.0063
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 282us/step - loss: 0.0087 - val_loss: 0.0064
Epoch 11/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 283us/step - loss: 0.0087 - val_loss: 0.0064


,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,"[32, 16, 8]",0.0015,16,200,0.0104,0.0066,0.1019,0.0812
2,"[32, 16, 8]",0.0015,16,200,0.0101,0.0063,0.1007,0.0791
3,"[32, 16, 8]",0.0015,16,200,0.0100,0.0062,0.1002,0.0789
4,"[32, 16, 8]",0.0015,16,200,0.0100,0.0062,0.1000,0.0787
5,"[32, 16, 8]",0.0015,16,200,0.0099,0.0063,0.0997,0.0791
6,"[32, 16, 8]",0.0015,16,200,0.0099,0.0062,0.0995,0.0785
7,"[32, 16, 8]",0.0015,16,200,0.0099,0.0064,0.0993,0.0797
8,"[32, 16, 8]",0.0015,16,200,0.0098,0.0064,0.0992,0.0803
9,"[32, 16, 8]",0.0015,16,200,0.0098,0.0063,0.0989,0.0796


Epoch 1/100


...

Epoch 1/100


/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 267us/step - loss: 0.2391 - val_loss: 0.0076
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 268us/step - loss: 0.0103 - val_loss: 0.0066
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 320us/step - loss: 0.0094 - val_loss: 0.0065
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 280us/step - loss: 0.0092 - val_loss: 0.0064
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 286us/step - loss: 0.0091 - val_loss: 0.0065
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 265us/step - loss: 0.0090 - val_loss: 0.0065
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 292us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 275us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 263us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 270us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 11/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 263us/step - loss: 0.0089 - val_loss: 0.0064
Epoch 12/

,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,[16],0.001,16,200,0.0678,0.0076,0.2605,0.0874
2,[16],0.001,16,200,0.0111,0.0066,0.1055,0.0810
3,[16],0.001,16,200,0.0105,0.0065,0.1023,0.0805
4,[16],0.001,16,200,0.0103,0.0064,0.1013,0.0801
5,[16],0.001,16,200,0.0101,0.0065,0.1007,0.0804
6,[16],0.001,16,200,0.0101,0.0065,0.1004,0.0803
7,[16],0.001,16,200,0.0101,0.0064,0.1003,0.0801
8,[16],0.001,16,200,0.0100,0.0064,0.1002,0.0798
9,[16],0.001,16,200,0.0100,0.0064,0.1002,0.0799


,Hidden Layers,Learning Rate,Batch Size,Seed,Epoch,Training MSE,Validation MSE,Training RMSE,Validation RMSE
0,"[16, 8]",0.0015,16,200,1,0.0225,0.0059,0.1501,0.0770
1,"[16, 8]",0.0005,16,200,2,0.0105,0.0059,0.1024,0.0771
2,"[16, 8]",0.0010,16,200,2,0.0104,0.0059,0.1019,0.0771
3,"[32, 16, 8]",0.0005,16,200,6,0.0099,0.0061,0.0993,0.0783
4,"[32, 16, 8]",0.0015,16,200,6,0.0099,0.0062,0.0995,0.0785
5,[16],0.0010,16,200,29,0.0099,0.0062,0.0994,0.0790
6,"[32, 16, 8]",0.0010,16,200,3,0.0100,0.0063,0.1000,0.0791
7,[32],0.0010,16,200,12,0.0099,0.0066,0.0996,0.0811


In [29]:
model_results = []
selected_model, selected_history, selected_error_table = run_network(
    [16, 8],
    0.0015
)

ols_predictions = ols_benchmark.predict(
    X_test.values
)

nn_predictions = selected_model.predict(
    X_test.values,
    verbose=0
).flatten()

test_results = test[
    ["date", "month", "permno", TARGET]
].copy()

test_results["OLS prediction"] = ols_predictions
test_results["NN prediction"] = nn_predictions

display(test_results.head())

Epoch 1/100


/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 276us/step - loss: 0.0754 - val_loss: 0.0059
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 264us/step - loss: 0.0092 - val_loss: 0.0059
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 267us/step - loss: 0.0092 - val_loss: 0.0064
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 269us/step - loss: 0.0090 - val_loss: 0.0066
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 264us/step - loss: 0.0089 - val_loss: 0.0067
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 266us/step - loss: 0.0089 - val_loss: 0.0067


,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,"[16, 8]",0.0015,16,200,0.0225,0.0059,0.1501,0.0770
2,"[16, 8]",0.0015,16,200,0.0104,0.0059,0.1019,0.0771
3,"[16, 8]",0.0015,16,200,0.0103,0.0064,0.1015,0.0803
4,"[16, 8]",0.0015,16,200,0.0102,0.0066,0.1008,0.0812
5,"[16, 8]",0.0015,16,200,0.0100,0.0067,0.1002,0.0816
6,"[16, 8]",0.0015,16,200,0.0100,0.0067,0.0999,0.0817


/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


,date,month,permno,target_ret_excess_tp1,OLS prediction,NN prediction
587,2019-01-31,2019-01,10104,0.0360,-0.0001,0.0124
588,2019-02-28,2019-02,10104,0.0284,0.0071,0.0124
589,2019-03-29,2019-03,10104,0.0325,0.0044,0.0124
590,2019-04-30,2019-04,10104,-0.0876,0.0097,0.0124
591,2019-05-31,2019-05,10104,0.1241,0.0106,0.0124
